# Quadratic $F_A$ reweighting validation

Appendix-quality checks of the z-expansion reweighting scheme. The seven `MaCCQE_UBGenie` weights of every event are samples of the CCQE cross-section ratio at $M_A = 0.8, 0.9, \dots, 1.4$ GeV; each knot is mapped to the dipole $F_A^{\mathrm{dip}}(Q^2_{\mathrm{true}}; M_A)$ and the seven samples are fit with $w(F_A) = c_0 + c_1 F_A + c_2 F_A^2$. At fixed kinematics the CCQE cross section is bilinear in the form factors, so the fit should be (near) exact.

Sample: the simulated $\nu_\mu$ CC1p events entering the PROfit fit (the `MCFile` selection of the XML configurations, weight `net_weight`).

Figures written below `figs/`:

- `quadratic_fa_residuals.pdf`: per-event maximum relative residual of the quadratic fit.
- `quadratic_fa_residual_vs_q2.pdf`: the same residual against $Q^2_{\mathrm{true}}$ (2D histogram with median, 68% central interval and 99th percentile profile).
- `quadratic_fa_binned_closure.pdf`: binned closure test, per-bin fractional difference between the spectra predicted with $w_{\mathrm{quad}}$ and with $w_{\mathrm{GENIE}}$ in the $\log_{10}Q^2_{\mathrm{reco}}$ and $p_n^{\mathrm{reco}}$ projections of the fit binning.
- `quadratic_fa_q2tilde_test.pdf`: diagnostic of the residual origin (form factors evaluated at the binding-energy-shifted $\tilde{Q}^2$ inside GENIE's Nieves model).
- `quadratic_fa_representative_event.pdf`: the quadratic fit through the seven spline samples of a median-residual event.

In [ ]:
from pathlib import Path
import sys

import awkward as ak
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

# Locate ma_zexp/python/scripts (shared style and figure root) and uboone_ngem
# (reweighting code) whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
ngem_repo = next(
    (parent / 'uboone_ngem' for parent in (start, *start.parents)
     if (parent / 'uboone_ngem' / 'src' / 'zexp_reweighting.py').is_file()),
    None,
)
if helper_dir is None or ngem_repo is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts and uboone_ngem/src')
for path in (helper_dir, ngem_repo):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from postfit_physical_parameters import FIGURE_ROOT, PUBLICATION_RC
from src.zexp_reweighting import (
    GENIE_DIPOLE_FA_Q2_ZERO, MA_CCQE_GRID_GEV, quadratic_fa_spline_weights,
    _clean_ma_spline_weights,
)

mpl.rcParams.update(PUBLICATION_RC)
np.set_printoptions(linewidth=150, precision=4, suppress=True)

INPUT_FILE = Path('/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root')
TREE_NAME = 'tree'

# Fit binning, copied from the <bins2D> element of the PROfit XML configurations.
Q2_LOG_EDGES = np.array([-2.00, -1.50, -1.20, -1.00, -0.85, -0.70, -0.55, -0.40, -0.20, 0.20])
PN_EDGES = np.array([0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00])

# "Responsive" tolerance: an event is responsive to M_A when its seven MaCCQE
# weights are not all identical, i.e. max_k w_k - min_k w_k > RESPONSIVE_TOLERANCE.
RESPONSIVE_TOLERANCE = 1e-5

SAVE_FIGURES = True
SAVE_DPI = 600
SAVE_FORMATS = ('pdf',)

# Okabe-Ito colours shared with the other publication figures.
COLOR_FIT = '#0072B2'
COLOR_HIST = '#56B4E9'
COLOR_GUIDE = '#D55E00'
COLOR_ALT = '#009E73'
COLOR_GRID = '#9AA4B2'
COLOR_NOTE = '#596273'
# Knot colours: diverging about the central knot (M_A = 1.1 GeV), blue below, orange above.
KNOT_COLORS = [plt.cm.Blues(0.95), plt.cm.Blues(0.72), plt.cm.Blues(0.50), '#6E6E6E',
               plt.cm.Oranges(0.50), plt.cm.Oranges(0.72), plt.cm.Oranges(0.95)]
KNOT_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

assert INPUT_FILE.is_file(), INPUT_FILE


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = FIGURE_ROOT / f'{stem}.{extension}'
        fig.savefig(path, dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0.03,
                    facecolor='white', format=extension)
        print('Saved:', path)


def style_axis(ax, log_x=False, log_y=False):
    ax.grid(which='major', color=COLOR_GRID, alpha=0.22, linewidth=0.7)
    if log_x:
        ax.grid(which='minor', axis='x', color=COLOR_GRID, alpha=0.10, linewidth=0.5)
    if log_y:
        ax.grid(which='minor', axis='y', color=COLOR_GRID, alpha=0.10, linewidth=0.5)
    ax.tick_params(which='both', direction='in', top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)


FIGURE_ROOT

## Fit sample and the definition of "responsive"

The sample is the simulated overlay component of the PROfit `MCFile` selection: `isdata==0 && isext==0 && isdirt==0 && isnuwro==0 && afro_1mu1p_sel==1`, weighted by `net_weight`, with finite reconstructed $Q^2 > 0$ and $p_n$. Every row carries a seven-knot `MaCCQE_UBGenie` spline.

An event is **responsive** when all of the following hold:

1. the seven weights are finite,
2. $Q^2_{\mathrm{true}}$ is finite and $\geq 0$,
3. the central-knot weight $w_3$ ($M_A = 1.1$ GeV) is positive,
4. $\max_k w_k - \min_k w_k > 10^{-5}$ (the weights are not all identical).

Non-responsive events carry no $M_A$ information: their quadratic "fit" is the constant $c_0 = w_3$, reproduces the seven weights exactly, and they are excluded from the per-event residual figures. They are kept in the binned closure test, where they enter every bin sum.

In [ ]:
BRANCHES = [
    'MaCCQE_UBGenie', 'GTruth_gQ2', 'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'afro_1mu1p_sel',
    'net_weight', 'isdata', 'isext', 'isdirt', 'isnuwro', 'wc_truth_nuScatType',
    'truth_nuEnergy', 'TunedCentralValue_UBGenie', 'RPA_CCQE_UBGenie',
]
with uproot.open(INPUT_FILE) as root_file:
    tree = root_file[TREE_NAME]
    n_rows_total = tree.num_entries
    raw = tree.arrays(BRANCHES, library='ak')

assert ak.all(ak.num(raw.MaCCQE_UBGenie) == len(MA_CCQE_GRID_GEV))
ma_weights_all = ak.to_numpy(raw.MaCCQE_UBGenie).astype(float)
q2_true_all = ak.to_numpy(raw.GTruth_gQ2).astype(float)
q2_reco_all = ak.to_numpy(raw.afro_1mu1p_Q2).astype(float)
pn_reco_all = ak.to_numpy(raw.afro_1mu1p_Pn).astype(float)
net_weight_all = ak.to_numpy(raw.net_weight).astype(float)
mode_all = np.nan_to_num(ak.to_numpy(raw.wc_truth_nuScatType), nan=-1).astype(int)
enu_all = ak.to_numpy(raw.truth_nuEnergy).astype(float)
tuned_cv_all = ak.to_numpy(raw.TunedCentralValue_UBGenie)[:, 0].astype(float)
rpa_all = ak.to_numpy(raw.RPA_CCQE_UBGenie).astype(float)

overlay = (~ak.to_numpy(raw.isdata) & ~ak.to_numpy(raw.isext)
           & ~ak.to_numpy(raw.isdirt) & ~ak.to_numpy(raw.isnuwro))
selected = overlay & ak.to_numpy(raw.afro_1mu1p_sel)
fit_sample = (selected & np.isfinite(q2_reco_all) & (q2_reco_all > 0)
              & np.isfinite(pn_reco_all) & np.isfinite(net_weight_all))

finite_weights = np.isfinite(ma_weights_all).all(axis=1)
weight_range = np.where(finite_weights, np.ptp(np.nan_to_num(ma_weights_all), axis=1), np.nan)
responsive_all = (
    finite_weights
    & np.isfinite(q2_true_all) & (q2_true_all >= 0)
    & (ma_weights_all[:, 3] > 0)
    & (weight_range > RESPONSIVE_TOLERANCE)
)
responsive = fit_sample & responsive_all

MODE_NAMES = {1: 'QE', 3: 'DIS', 4: 'RES', 5: 'COH', 10: 'MEC'}
def mode_counts(mask):
    values, counts = np.unique(mode_all[mask], return_counts=True)
    return ', '.join(f'{MODE_NAMES.get(v, v)}: {c:,}' for v, c in zip(values, counts))

n_selected = int(fit_sample.sum())
n_responsive = int(responsive.sum())
nonresponsive = fit_sample & ~responsive_all
nonzero_range = weight_range[fit_sample & finite_weights]
nonzero_range = nonzero_range[nonzero_range > 0]

print(f'Tree rows: {n_rows_total:,}; simulated overlay rows: {overlay.sum():,}')
print(f'Selected simulated 1mu1p events (fit sample): {n_selected:,}')
print(f'Responsive events (weight range > {RESPONSIVE_TOLERANCE:g}): {n_responsive:,} '
      f'({100 * n_responsive / n_selected:.2f}% of the fit sample)')
print(f'  responsive events by interaction mode: {mode_counts(responsive)}')
print(f'Non-responsive events: {nonresponsive.sum():,}')
print(f'  all seven weights identical (range = 0): {(nonresponsive & (weight_range == 0)).sum():,} '
      f'[{mode_counts(nonresponsive & (weight_range == 0))}]')
print(f'  0 < range <= tolerance: {(nonresponsive & (weight_range > 0) & (weight_range <= RESPONSIVE_TOLERANCE)).sum():,}')
print(f'  central weight w_3 <= 0: {(nonresponsive & finite_weights & ~(ma_weights_all[:, 3] > 0)).sum():,} '
      f'(all seven weights are exactly zero for these RES events)')
print(f'  non-finite weights or unphysical true Q^2: '
      f'{(nonresponsive & (~finite_weights | ~(np.isfinite(q2_true_all) & (q2_true_all >= 0)))).sum():,}')
print(f'Smallest non-zero weight range in the fit sample: {nonzero_range.min():.3g} '
      f'-> the responsive count is insensitive to any tolerance below that value.')
print(f'Central-knot weight equals TunedCentralValue_UBGenie for '
      f'{100 * np.mean(np.isclose(ma_weights_all[responsive, 3], tuned_cv_all[responsive])):.1f}% of responsive events '
      f'(the GENIE knob weights include the tune CV weight).')

q2_true = q2_true_all[responsive]
ma_weights = ma_weights_all[responsive]
print(f'true Q^2 range of responsive events: {q2_true.min():.4g} to {q2_true.max():.4g} GeV^2')

## Per-event residuals of the quadratic fit at the seven spline samples

Each responsive event's seven weights are fit by a quadratic in $F_A^{\mathrm{dip}}(Q^2_{\mathrm{true}}; M_A)$ and the fit is evaluated back at the seven knots. The per-event figure of merit is

$$r = \frac{\max_k |w_{\mathrm{quad}}(F_A^{(k)}) - w_k|}{\max_k |w_k|}.$$

The denominator is $\max_k |w_k|$ rather than the per-knot $w_k$: the GENIE weights of a single event can pass through zero (the CCQE cross section has a zero in $F_A$ at fixed $Q^2$, and events near the RPA suppression region carry weights close to zero at some knots), so a per-knot relative residual $|w_{\mathrm{quad}} - w_k| / |w_k|$ would blow up for reasons unrelated to the quality of the fit. Normalizing by the largest weight of the event gives the residual relative to the scale of the event's weight, which is what matters when the weights are summed into bins.

In [ ]:
def quadratic_fit_residuals(q2, weights, q2_shift=0.0):
    # Refit the seven knots with the dipole F_A evaluated at q2 + q2_shift.
    # Returns the fitted weights at the seven knots and the per-event maximum
    # relative residual max_k |w_quad - w_k| / max_k |w_k|.
    q2_eff = q2 + q2_shift
    fa_grid = GENIE_DIPOLE_FA_Q2_ZERO / (1 + q2_eff[:, None] / MA_CCQE_GRID_GEV[None, :]**2)**2
    fitted = np.column_stack([
        quadratic_fa_spline_weights(q2_eff, fa_grid[:, i], weights)
        for i in range(len(MA_CCQE_GRID_GEV))
    ])
    residual = np.max(np.abs(fitted - weights), axis=1) / np.maximum(
        np.max(np.abs(weights), axis=1), 1e-12)
    return fitted, residual


fa_grid = GENIE_DIPOLE_FA_Q2_ZERO / (1 + q2_true[:, None] / MA_CCQE_GRID_GEV[None, :]**2)**2
quadratic_at_grid, max_relative_residual = quadratic_fit_residuals(q2_true, ma_weights)
signed_residual = (quadratic_at_grid - ma_weights) / np.max(np.abs(ma_weights), axis=1)[:, None]

percentile_levels = [50, 68, 90, 95, 99, 99.9, 100]
residual_percentiles = pd.DataFrame({
    'percentile': percentile_levels,
    'max relative residual [%]': 100 * np.percentile(max_relative_residual, percentile_levels),
}).set_index('percentile')
display(residual_percentiles.style.format({'max relative residual [%]': '{:.5f}'}))

median_residual = np.median(max_relative_residual)
p99_residual = np.percentile(max_relative_residual, 99)
worst = int(np.argmax(max_relative_residual))
print(f'Median max. relative residual: {100 * median_residual:.4f}%   99th percentile: {100 * p99_residual:.4f}%')
print(f'Worst event: residual {max_relative_residual[worst]:.3e} '
      f'({100 * max_relative_residual[worst]:.3f}%)')
print(f'Fraction of responsive events with residual > 1e-3: {np.mean(max_relative_residual > 1e-3):.2e} '
      f'({np.sum(max_relative_residual > 1e-3)} events)')

# Signed residual pattern across the knots: the quadratic misses the seven
# samples with a fixed alternating pattern, which is the signature of a smooth
# systematic deviation from a quadratic rather than of noise.
pattern = pd.DataFrame({
    'M_A [GeV]': MA_CCQE_GRID_GEV,
    'median signed residual [1e-5]': 1e5 * np.median(signed_residual, axis=0),
    'fraction of events where this knot is the worst': np.bincount(
        np.argmax(np.abs(signed_residual), axis=1), minlength=7) / len(signed_residual),
}).set_index('M_A [GeV]')
display(pattern.style.format('{:.3f}'))

### The largest residuals

The events with the largest residuals are listed with their kinematics and weights. The last block quantifies what they have in common by correlating the residual, at fixed $Q^2_{\mathrm{true}}$, with the dynamic range of the seven weights, with the tune central-value weight, and with the neutrino energy.

In [ ]:
idx_responsive = np.where(responsive)[0]
order = np.argsort(max_relative_residual)[::-1]
rows = []
for j in order[:12]:
    i = idx_responsive[j]
    rows.append({
        'residual': max_relative_residual[j],
        'Q2_true [GeV2]': q2_true[j],
        'Q2_reco [GeV2]': q2_reco_all[i],
        'p_n reco [GeV]': pn_reco_all[i],
        'mode': MODE_NAMES.get(mode_all[i], mode_all[i]),
        'E_nu [MeV]': enu_all[i],
        'w_3 (= tune CV)': ma_weights[j, 3],
        'net_weight': net_weight_all[i],
        'w_0/w_3': ma_weights[j, 0] / ma_weights[j, 3],
        'w_6/w_3': ma_weights[j, 6] / ma_weights[j, 3],
        'min RPA knob weight': rpa_all[i].min(),
    })
worst_table = pd.DataFrame(rows)
display(worst_table.style.format({'residual': '{:.2e}', 'Q2_true [GeV2]': '{:.3f}', 'Q2_reco [GeV2]': '{:.3f}',
                                  'p_n reco [GeV]': '{:.3f}', 'E_nu [MeV]': '{:.0f}', 'w_3 (= tune CV)': '{:.3f}',
                                  'net_weight': '{:.4f}', 'w_0/w_3': '{:.3f}', 'w_6/w_3': '{:.3f}',
                                  'min RPA knob weight': '{:.3f}'}))

print('Seven weights of the worst event:', ma_weights[worst])
print('Quadratic fit at the seven knots :', quadratic_at_grid[worst])

dynamic_range = np.ptp(ma_weights, axis=1) / ma_weights[:, 3]
print('\nSpearman rank correlation of the residual with event properties, in slices of true Q^2:')
def spearman(x, y):
    rx = pd.Series(x).rank().to_numpy(); ry = pd.Series(y).rank().to_numpy()
    return np.corrcoef(rx, ry)[0, 1]
corr_rows = []
for lo, hi in [(0.05, 0.15), (0.15, 0.3), (0.3, 0.6), (0.6, 1.2), (1.2, 10.0)]:
    s = (q2_true >= lo) & (q2_true < hi)
    corr_rows.append({
        'Q2_true slice [GeV2]': f'[{lo:g}, {hi:g})', 'events': int(s.sum()),
        'rho(residual, Q2_true)': spearman(max_relative_residual[s], q2_true[s]),
        'rho(residual, weight dynamic range)': spearman(max_relative_residual[s], dynamic_range[s]),
        'rho(residual, tune CV weight)': spearman(max_relative_residual[s], ma_weights[s, 3]),
        'rho(residual, E_nu)': spearman(max_relative_residual[s], enu_all[idx_responsive][s]),
    })
display(pd.DataFrame(corr_rows).set_index('Q2_true slice [GeV2]').style.format('{:.2f}', subset=pd.IndexSlice[:, [c for c in corr_rows[0] if c.startswith('rho')]]))

### Comparison with the earlier pre-selection sample

The first version of this check used the leading 100,000 rows of the tree without the $1\mu1p$ selection. Those numbers are reproduced here so that the two samples can be compared; the tail of that sample (residuals of $3\times10^{-3}$ to $5\times10^{-3}$) is examined explicitly.

In [ ]:
leading = np.zeros(len(ma_weights_all), dtype=bool)
leading[:100_000] = True
leading &= responsive_all
_, leading_residual = quadratic_fit_residuals(q2_true_all[leading], ma_weights_all[leading])
print(f'Leading 100,000 rows (no selection): {leading.sum():,} responsive events; '
      f'median {100 * np.median(leading_residual):.4f}%, 99th pct {100 * np.percentile(leading_residual, 99):.4f}%, '
      f'max {leading_residual.max():.2e}')
idx_leading = np.where(leading)[0]
tail = np.argsort(leading_residual)[::-1][:6]
tail_table = pd.DataFrame({
    'residual': leading_residual[tail],
    'Q2_true [GeV2]': q2_true_all[idx_leading[tail]],
    'mode': [MODE_NAMES.get(m, m) for m in mode_all[idx_leading[tail]]],
    'E_nu [MeV]': enu_all[idx_leading[tail]],
    'w_3 (= tune CV)': ma_weights_all[idx_leading[tail], 3],
    'w_0/w_3': ma_weights_all[idx_leading[tail], 0] / ma_weights_all[idx_leading[tail], 3],
    'w_6/w_3': ma_weights_all[idx_leading[tail], 6] / ma_weights_all[idx_leading[tail], 3],
    'min RPA knob weight': rpa_all[idx_leading[tail]].min(axis=1),
    'passes 1mu1p selection': fit_sample[idx_leading[tail]],
})
display(tail_table.style.format({'residual': '{:.2e}', 'Q2_true [GeV2]': '{:.3f}', 'E_nu [MeV]': '{:.0f}',
                                 'w_3 (= tune CV)': '{:.3f}', 'w_0/w_3': '{:.3f}', 'w_6/w_3': '{:.3f}',
                                 'min RPA knob weight': '{:.3f}'}))

## Figure 1: residual distribution

In [ ]:
positive = max_relative_residual[max_relative_residual > 0]
bins = np.geomspace(1e-7, max(positive.max() * 1.5, 1e-6), 60)

fig, ax = plt.subplots(figsize=(7.0, 5.0), constrained_layout=True)
counts, _, _ = ax.hist(positive, bins=bins, histtype='stepfilled',
                       facecolor=mpl.colors.to_rgba(COLOR_HIST, 0.45), edgecolor=COLOR_FIT,
                       linewidth=1.4, zorder=3)
ax.axvline(median_residual, color=COLOR_GUIDE, linestyle='--', linewidth=1.3, zorder=4,
           label=f'Median: {100 * median_residual:.4f}%')
ax.axvline(p99_residual, color=COLOR_GUIDE, linestyle=':', linewidth=1.3, zorder=4,
           label=f'99th percentile: {100 * p99_residual:.4f}%')
ax.set(xscale='log', yscale='log')
ax.set_xlabel(r'$\max_k\,|w_{\mathrm{quad}}(F_A^{(k)}) - w_k| / \max_k |w_k|$')
ax.set_ylabel('Events')
ax.set_xlim(bins[0], bins[-1])
ax.set_ylim(0.7, counts.max() * 6)  # headroom for the legend and note
ax.legend(loc='upper left', fontsize=12)
ax.text(0.03, 0.80, f'Worst event: {max_relative_residual[worst]:.1e}',
        transform=ax.transAxes, ha='left', va='top', fontsize=12, color=COLOR_NOTE)
style_axis(ax, log_x=True)

save_figure(fig, 'quadratic_fa_residuals')
plt.show()

## Figure 2: residual versus $Q^2_{\mathrm{true}}$

2D histogram of the per-event residual against $Q^2_{\mathrm{true}}$, with the median, the 68% central interval (16th to 84th percentile) and the 99th percentile of the residual in bins of $Q^2_{\mathrm{true}}$. The printed table gives the same profile and a power-law fit to the median.

In [ ]:
q2_edges = np.geomspace(2e-3, 4.0, 24)
res_edges = np.geomspace(1e-7, 3e-3, 44)
q2_centers = np.sqrt(q2_edges[:-1] * q2_edges[1:])

profile = []
for lo, hi in zip(q2_edges[:-1], q2_edges[1:]):
    s = (q2_true >= lo) & (q2_true < hi) & (max_relative_residual > 0)
    if s.sum() >= 10:
        r = max_relative_residual[s]
        profile.append((np.sqrt(lo * hi), s.sum(), *np.percentile(r, [16, 50, 84, 99]), r.max()))
    else:
        profile.append((np.sqrt(lo * hi), s.sum(), *(np.nan,) * 5))
profile = pd.DataFrame(profile, columns=['Q2_true center [GeV2]', 'events', 'p16', 'median', 'p84', 'p99', 'max'])
display(profile.style.format({'Q2_true center [GeV2]': '{:.4f}', 'p16': '{:.2e}', 'median': '{:.2e}',
                              'p84': '{:.2e}', 'p99': '{:.2e}', 'max': '{:.2e}'}))

good = profile['events'] >= 50
slope, intercept = np.polyfit(np.log(profile.loc[good, 'Q2_true center [GeV2]']),
                              np.log(profile.loc[good, 'median']), 1)
rank_corr = spearman(np.log(max_relative_residual + 1e-12), np.log(q2_true))
print(f'Median residual scales as Q2_true^{slope:.2f} over the bins with >= 50 events '
      f'(median = {np.exp(intercept):.2e} x (Q2_true / GeV^2)^{slope:.2f}).')
print(f'Spearman rank correlation between residual and Q2_true: {rank_corr:.2f}')
low = q2_true < 0.1; high = q2_true > 0.5
print(f'Median residual for Q2_true < 0.1 GeV^2: {np.median(max_relative_residual[low]):.2e}  '
      f'({low.sum():,} events); for Q2_true > 0.5 GeV^2: {np.median(max_relative_residual[high]):.2e}  ({high.sum():,} events)')

fig, ax = plt.subplots(figsize=(7.0, 5.0), constrained_layout=True)
hist2d, _, _ = np.histogram2d(q2_true, np.clip(max_relative_residual, res_edges[0], None),
                              bins=(q2_edges, res_edges))
mesh = ax.pcolormesh(q2_edges, res_edges, hist2d.T, cmap='Blues',
                     norm=LogNorm(vmin=1, vmax=hist2d.max()), rasterized=True, zorder=2)
cbar = fig.colorbar(mesh, ax=ax, pad=0.02, aspect=30)
cbar.set_label('Events', labelpad=8)
cbar.ax.tick_params(which='both', direction='in')
ok = profile['events'] >= 10
xs = profile.loc[ok, 'Q2_true center [GeV2]'].to_numpy()
# The last populated bin extends past Q^2 = 2 GeV^2; its profile is drawn flat out to
# Q^2 = 2 so the curves span the populated range instead of ending at the bin centre.
xs = np.append(xs, min(2.0, q2_edges[1:][ok.to_numpy()][-1]))

def profile_curve(column):
    values = profile.loc[ok, column].to_numpy()
    return np.append(values, values[-1])

ax.fill_between(xs, profile_curve('p16'), profile_curve('p84'), color=COLOR_GUIDE, alpha=0.22,
                linewidth=0, zorder=3, label='68% central interval')
ax.plot(xs, profile_curve('median'), color=COLOR_GUIDE, linewidth=1.8, zorder=4, label='Median')
ax.plot(xs, profile_curve('p99'), color=COLOR_GUIDE, linewidth=1.4, linestyle=':', zorder=4,
        label='99th percentile')
ax.set(xscale='log', yscale='log')
ax.set_xlim(q2_edges[0], q2_edges[-1])
ax.set_ylim(res_edges[0], res_edges[-1])
ax.set_xlabel(r'$Q^2_{\mathrm{true}}$ [GeV$^2$]')
ax.set_ylabel(r'Maximum relative residual per event')
ax.legend(loc='upper left', fontsize=12)
ax.text(0.97, 0.04, rf'median $\propto (Q^2_{{\mathrm{{true}}}})^{{{slope:.2f}}}$',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=12, color=COLOR_NOTE)
style_axis(ax, log_x=True, log_y=True)

save_figure(fig, 'quadratic_fa_residual_vs_q2')
plt.show()

## Figure 3: binned closure test

For each of the seven knots the predicted spectrum of the fit sample is built twice, weighting every selected event with `net_weight` $\times\, w_{\mathrm{GENIE}}(M_A)$ and with `net_weight` $\times\, w_{\mathrm{quad}}(F_A^{\mathrm{dip}}(M_A))$, in the $9\times 8$ $(\log_{10}Q^2_{\mathrm{reco}}, p_n^{\mathrm{reco}})$ fit bins and in their two projections. Non-responsive events are included (their quadratic reproduces the GENIE weight exactly); non-finite weights are reset to one exactly as in the production code. The figure shows the per-bin fractional difference $(S_{\mathrm{quad}} - S_{\mathrm{GENIE}})/S_{\mathrm{GENIE}}$ of the two projections; the table reports the largest absolute difference per knot for the 2D bins and both projections, and for the PROfit-style ratio to the central knot (`force_0_cv`), which is how the spline actually enters the fit.

In [ ]:
fit_q2_true = q2_true_all[fit_sample]
fit_ma_weights = _clean_ma_spline_weights(ma_weights_all[fit_sample])
fit_fa_grid = GENIE_DIPOLE_FA_Q2_ZERO / (1 + fit_q2_true[:, None] / MA_CCQE_GRID_GEV[None, :]**2)**2
fit_quadratic = np.column_stack([
    quadratic_fa_spline_weights(fit_q2_true, fit_fa_grid[:, i], fit_ma_weights)
    for i in range(len(MA_CCQE_GRID_GEV))
])
log_q2_reco = np.log10(q2_reco_all[fit_sample])
pn_reco = pn_reco_all[fit_sample]
event_weight = net_weight_all[fit_sample]
in_range = ((log_q2_reco >= Q2_LOG_EDGES[0]) & (log_q2_reco < Q2_LOG_EDGES[-1])
            & (pn_reco >= PN_EDGES[0]) & (pn_reco < PN_EDGES[-1]))
print(f'{in_range.sum():,} of {n_selected:,} selected events fall inside the fit binning '
      f'(expected yield {event_weight[in_range].sum():,.1f} events at the MC POT)')

def spectrum_2d(knob_weights):
    counts, _, _ = np.histogram2d(log_q2_reco, pn_reco, bins=(Q2_LOG_EDGES, PN_EDGES),
                                  weights=event_weight * knob_weights)
    return counts

spectra_genie = np.array([spectrum_2d(fit_ma_weights[:, k]) for k in range(7)])
spectra_quad = np.array([spectrum_2d(fit_quadratic[:, k]) for k in range(7)])

closure_2d = spectra_quad / spectra_genie - 1
closure_q2 = spectra_quad.sum(axis=2) / spectra_genie.sum(axis=2) - 1
closure_pn = spectra_quad.sum(axis=1) / spectra_genie.sum(axis=1) - 1
# PROfit force_0_cv: every knot spectrum is divided by the knot-0 (M_A = 1.1 GeV) spectrum.
closure_2d_ratio = (spectra_quad / spectra_quad[3]) / (spectra_genie / spectra_genie[3]) - 1
ma_effect = np.abs(spectra_genie / spectra_genie[3] - 1)

closure_table = pd.DataFrame({
    'M_A [GeV]': MA_CCQE_GRID_GEV,
    'max |dS/S| 2D fit bins': np.abs(closure_2d).max(axis=(1, 2)),
    'max |dS/S| log10 Q2_reco proj.': np.abs(closure_q2).max(axis=1),
    'max |dS/S| p_n reco proj.': np.abs(closure_pn).max(axis=1),
    'max |dS/S| 2D, ratio to central knot': np.abs(closure_2d_ratio).max(axis=(1, 2)),
    'max |S_k/S_3 - 1| 2D (size of the M_A effect)': ma_effect.max(axis=(1, 2)),
}).set_index('M_A [GeV]')
display(closure_table.style.format('{:.2e}'))

worst_bin = np.unravel_index(np.argmax(np.abs(closure_2d)), closure_2d.shape)
print(f'Largest absolute per-bin fractional difference over all 72 fit bins and 7 knots: '
      f'{np.abs(closure_2d).max():.2e}  (M_A = {MA_CCQE_GRID_GEV[worst_bin[0]]:.1f} GeV, '
      f'log10 Q2_reco bin [{Q2_LOG_EDGES[worst_bin[1]]:.2f}, {Q2_LOG_EDGES[worst_bin[1] + 1]:.2f}), '
      f'p_n bin [{PN_EDGES[worst_bin[2]]:.2f}, {PN_EDGES[worst_bin[2] + 1]:.2f}) GeV)')
print(f'Largest over the log10 Q2_reco projection: {np.abs(closure_q2).max():.2e}; '
      f'over the p_n projection: {np.abs(closure_pn).max():.2e}')
print(f'Largest for the PROfit-style ratio to the central knot (2D bins): {np.abs(closure_2d_ratio).max():.2e}')
print(f'For scale, the M_A variation itself changes the 2D bins by up to {ma_effect.max():.2f} (fractional) at the outer knots.')

In [ ]:
SCALE = 1e-4
fig, axes = plt.subplots(2, 1, figsize=(7.0, 8.4), constrained_layout=True)
panels = [
    (axes[0], Q2_LOG_EDGES, closure_q2, r'$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$'),
    (axes[1], PN_EDGES, closure_pn, r'$p_n^{\mathrm{reco}}$ [GeV/$c$]'),
]
for ax, edges, closure, xlabel in panels:
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    ax.axhline(0, color='black', linewidth=0.9, zorder=2)
    for k in range(7):
        # small horizontal offsets keep the seven markers of a bin legible
        offset = (k - 3) * 0.09 * widths
        ax.plot(centers + offset, closure[k] / SCALE, linestyle='none', marker=KNOT_MARKERS[k],
                markersize=6.0, color=KNOT_COLORS[k], markeredgecolor='black', markeredgewidth=0.4,
                zorder=4, label=rf'$M_A = {MA_CCQE_GRID_GEV[k]:.1f}$ GeV')
    for edge in edges[1:-1]:
        ax.axvline(edge, color=COLOR_GRID, alpha=0.35, linewidth=0.7, zorder=1)
    ax.set_xlim(edges[0], edges[-1])
    ax.set_xticks(edges)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r'$(S_{\mathrm{quad}} - S_{\mathrm{GENIE}})\,/\,S_{\mathrm{GENIE}}$  [$10^{-4}$]')
    ymax = 1.3 * np.abs(closure).max() / SCALE
    ax.set_ylim(-ymax, ymax)
    style_axis(ax)
    ax.grid(False, axis='x')
axes[0].tick_params(axis='x', labelrotation=35)
axes[0].legend(loc='upper left', ncol=2, fontsize=12, handletextpad=0.3, columnspacing=1.0)

save_figure(fig, 'quadratic_fa_binned_closure')
plt.show()

## Where the residual comes from

**Reweight configuration.** The `MaCCQE_UBGenie` weights are produced by MicroBooNE's `EventWeight` with the GENIE ReWeight dial `MaCCQE` ("MaCCQE uses +/- 0.10 GeV about the tuned central value", `ubsim/EventWeight/jobs/genie/eventweight_microboone_genie_single_knobs.fcl`). In `GReWeightNuXSecCCQE` this dial runs in `kModeMa`, whose `CalcWeightMa` returns `old_weight * new_xsec / old_xsec` with both cross sections the *differential* cross section at the event's kinematics; only the `MaCCQEshape` dial (`kModeNormAndMaShape`, `CalcWeightMaShape`) multiplies in the ratio of integrated cross sections. There is therefore no total-cross-section normalization factor in our weights. This is confirmed empirically below: as $Q^2_{\mathrm{true}} \to 0$ the dipole $F_A$ of all knots coincide, and the weights of the outer knots go to one, whereas a normalization factor would leave them at $\sigma(1.1\,\mathrm{GeV})/\sigma(M_A) \approx 0.85$ to $1.15$.

**The actual origin.** GENIE's Nieves CCQE model evaluates the form factors not at the true four-momentum transfer but at the de Forest-shifted $\tilde{Q}^2 = -\tilde{q}^2$, with $\tilde{q}^0 = E_{N'} - E_N^{\mathrm{on\,shell}}$ (`NievesQELCCPXSec::XSec`: `interaction->KinePtr()->SetQ2(Q2tilde); fFormFactors.Calculate(interaction); ... SetQ2(Q2)`), so $\tilde{Q}^2 = Q^2 + (q^0)^2 - (\tilde{q}^0)^2 \approx Q^2 + 2 q^0 \epsilon_{\mathrm{rem}}$. The weight of an event is exactly quadratic in $F_A(\tilde{Q}^2; M_A)$; expressed in $F_A(Q^2_{\mathrm{true}}; M_A)$ it is quadratic only to the extent that the two variables are linearly related across the seven knots. The test: refit every event with a free additive shift $\delta$, $F_A(Q^2_{\mathrm{true}} + \delta; M_A)$, and check (i) whether the residual collapses far below what one extra parameter would buy against noise and (ii) whether the fitted $\delta$ is positive, of order $2 q^0 \epsilon_{\mathrm{rem}}$ (tens of MeV$^2$ at low $Q^2$, growing linearly with $Q^2$ because $q^0 \approx Q^2 / 2M + \ldots$). The event kinematics needed to compute $\tilde{Q}^2$ directly are not in the tree, so this is the strongest test available here. Because the refit reaches the numerical floor of the stored weights, the one-parameter scan can have a second, equally deep minimum at a much larger shift for a few percent of events; the local minimum with the smallest shift is used.

In [ ]:
# (a) Low-Q^2 limit: outer-knot weights relative to the central knot.
ratio_to_cv = ma_weights / ma_weights[:, 3][:, None]
low_q2_rows = []
for lo, hi in [(0, 0.003), (0.003, 0.006), (0.006, 0.01), (0.01, 0.02), (0.02, 0.05), (0.3, 1.0)]:
    s = (q2_true >= lo) & (q2_true < hi)
    low_q2_rows.append({'Q2_true [GeV2]': f'[{lo:g}, {hi:g})', 'events': int(s.sum()),
                        **{f'median w(M_A={m:.1f})/w(1.1)': v for m, v in zip(MA_CCQE_GRID_GEV, np.median(ratio_to_cv[s], axis=0))}})
print('Low-Q^2 limit of the knob weights (a total-cross-section normalization would keep the outer knots ~0.85 to ~1.15):')
display(pd.DataFrame(low_q2_rows).set_index('Q2_true [GeV2]').style.format('{:.4f}', subset=pd.IndexSlice[:, [c for c in low_q2_rows[0] if c.startswith('median')]]))

# (b) Refit with a free additive shift of Q^2 (coarse scan, then a fine scan around the minimum).
coarse_shifts = np.linspace(-0.05, 0.30, 351)
scan = np.empty((len(coarse_shifts), n_responsive))
for k, shift in enumerate(coarse_shifts):
    scan[k] = quadratic_fit_residuals(q2_true, ma_weights, shift)[1]
# The residual floor (~1e-6) makes the one-parameter refit degenerate at large
# shifts: about 5% of events have a second, equally deep local minimum at
# delta ~ 0.1 to 0.3 GeV^2. Take the local minimum with the smallest shift.
is_local_min = np.zeros_like(scan, dtype=bool)
is_local_min[1:-1] = (scan[1:-1] < scan[:-2]) & (scan[1:-1] <= scan[2:])
has_local_min = is_local_min.any(axis=0)
best = np.where(has_local_min, np.argmax(is_local_min, axis=0), np.argmin(scan, axis=0))
fine_offsets = np.linspace(-0.0015, 0.0015, 61)
fine = np.empty((len(fine_offsets), n_responsive))
for k, offset in enumerate(fine_offsets):
    fine[k] = quadratic_fit_residuals(q2_true, ma_weights, coarse_shifts[best] + offset)[1]
best_fine = np.argmin(fine, axis=0)
q2_shift = coarse_shifts[best] + fine_offsets[best_fine]
shifted_residual = fine[best_fine, np.arange(n_responsive)]
improvement = max_relative_residual / np.maximum(shifted_residual, 1e-15)

shift_table = pd.DataFrame({
    'percentile': [50, 90, 99, 99.9, 100],
    'residual, F_A(Q2_true) [%]': 100 * np.percentile(max_relative_residual, [50, 90, 99, 99.9, 100]),
    'residual, F_A(Q2_true + delta) [%]': 100 * np.percentile(shifted_residual, [50, 90, 99, 99.9, 100]),
}).set_index('percentile')
display(shift_table.style.format('{:.6f}'))
print(f'Residual reduction factor: median {np.median(improvement):.0f}, '
      f'1st to 99th percentile {np.percentile(improvement, 1):.1f} to {np.percentile(improvement, 99):.0f}; '
      f'{100 * np.mean(improvement > 10):.1f}% of events improve by more than a factor 10')
print(f'{100 * np.mean(has_local_min):.2f}% of events have a local minimum inside the scan; '
      f'{100 * np.mean(is_local_min.sum(axis=0) > 1):.1f}% have more than one (the smallest shift is used).')
print(f'Fitted shift delta = Q2_eff - Q2_true: median {np.median(q2_shift):.4f} GeV^2, '
      f'16th to 84th percentile {np.percentile(q2_shift, 16):.4f} to {np.percentile(q2_shift, 84):.4f} GeV^2; '
      f'{100 * np.mean(q2_shift > 0):.2f}% positive; {np.mean(best == 0) + np.mean(best == len(coarse_shifts) - 1):.1e} at the scan edges')
lin = np.polyfit(q2_true, q2_shift, 1)
print(f'Linear fit: delta = {lin[1]:.4f} GeV^2 + {lin[0]:.4f} x Q2_true '
      f'(de Forest expectation: delta ~ 2 q0 eps_rem with q0 ~ Q2/2M, i.e. slope ~ eps_rem/M ~ 0.03 and a positive intercept)')

shift_profile = []
for lo, hi in zip(q2_edges[:-1], q2_edges[1:]):
    s = (q2_true >= lo) & (q2_true < hi)
    if s.sum() >= 20:
        shift_profile.append((np.sqrt(lo * hi), s.sum(), *np.percentile(q2_shift[s], [16, 50, 84]),
                              np.median(max_relative_residual[s]), np.median(shifted_residual[s])))
shift_profile = pd.DataFrame(shift_profile, columns=['Q2_true center [GeV2]', 'events', 'delta p16', 'delta median', 'delta p84',
                                                     'median residual before', 'median residual after'])
display(shift_profile.style.format({'Q2_true center [GeV2]': '{:.4f}', 'delta p16': '{:.4f}', 'delta median': '{:.4f}',
                                    'delta p84': '{:.4f}', 'median residual before': '{:.2e}', 'median residual after': '{:.2e}'}))
print('Worst event of the fit sample after the shift refit:', f'{shifted_residual[worst]:.2e}',
      f'(before {max_relative_residual[worst]:.2e}, delta = {q2_shift[worst]:.4f} GeV^2)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.8), constrained_layout=True)

ax = axes[0]
bins_shift = np.geomspace(1e-8, 3e-3, 66)
ax.hist(np.clip(max_relative_residual, bins_shift[0], None), bins=bins_shift, histtype='stepfilled',
        facecolor=mpl.colors.to_rgba(COLOR_HIST, 0.45), edgecolor=COLOR_FIT, linewidth=1.4, zorder=3,
        label=r'quadratic in $F_A(Q^2_{\mathrm{true}})$')
ax.hist(np.clip(shifted_residual, bins_shift[0], None), bins=bins_shift, histtype='step',
        edgecolor=COLOR_ALT, linewidth=1.6, zorder=4,
        label=r'quadratic in $F_A(Q^2_{\mathrm{true}} + \delta)$, $\delta$ fit per event')
ax.set(xscale='log', yscale='log')
ax.set_xlim(bins_shift[0], bins_shift[-1])
ax.set_ylim(0.7, 60 * max(np.histogram(shifted_residual, bins=bins_shift)[0].max(),
                          np.histogram(max_relative_residual, bins=bins_shift)[0].max()))
ax.set_xlabel('Maximum relative residual per event')
ax.set_ylabel('Events')
ax.legend(loc='upper left', fontsize=10.5)
ax.text(0.03, 0.80, f'median: {np.median(max_relative_residual):.1e} $\\to$ {np.median(shifted_residual):.1e}\n'
        f'99th pct.: {np.percentile(max_relative_residual, 99):.1e} $\\to$ {np.percentile(shifted_residual, 99):.1e}',
        transform=ax.transAxes, ha='left', va='top', fontsize=10.5, color=COLOR_NOTE)
style_axis(ax, log_x=True)

ax = axes[1]
hist_shift, _, _ = np.histogram2d(q2_true, np.clip(q2_shift, 1e-3, 0.3), bins=(q2_edges, np.geomspace(1e-3, 0.3, 40)))
mesh = ax.pcolormesh(q2_edges, np.geomspace(1e-3, 0.3, 40), hist_shift.T, cmap='Blues',
                     norm=LogNorm(vmin=1, vmax=hist_shift.max()), rasterized=True, zorder=2)
cbar = fig.colorbar(mesh, ax=ax, pad=0.02, aspect=30)
cbar.set_label('Events', labelpad=8)
cbar.ax.tick_params(which='both', direction='in')
ax.fill_between(shift_profile['Q2_true center [GeV2]'], shift_profile['delta p16'], shift_profile['delta p84'],
                color=COLOR_GUIDE, alpha=0.22, linewidth=0, zorder=3, label='68% central interval')
ax.plot(shift_profile['Q2_true center [GeV2]'], shift_profile['delta median'], color=COLOR_GUIDE, linewidth=1.8,
        zorder=4, label='Median')
q2_line = np.geomspace(q2_edges[0], q2_edges[-1], 100)
ax.plot(q2_line, lin[1] + lin[0] * q2_line, color='black', linestyle='--', linewidth=1.2, zorder=4,
        label=rf'$\delta = {lin[1]:.3f} + {lin[0]:.3f}\,Q^2_{{\mathrm{{true}}}}$')
ax.set(xscale='log', yscale='log')
ax.set_xlim(q2_edges[0], q2_edges[-1])
ax.set_ylim(1e-3, 0.3)
ax.set_xlabel(r'$Q^2_{\mathrm{true}}$ [GeV$^2$]')
ax.set_ylabel(r'Fitted shift $\delta = \tilde{Q}^2 - Q^2_{\mathrm{true}}$ [GeV$^2$]')
ax.legend(loc='lower right', fontsize=10.5)
style_axis(ax, log_x=True, log_y=True)

save_figure(fig, 'quadratic_fa_q2tilde_test')
plt.show()

## Figure 5: representative event

The event with the median maximum relative residual. The seven GENIE spline samples sit on the fitted quadratic; the largest deviation for this event is quoted on the panel.

In [ ]:
example_i = int(np.argsort(max_relative_residual)[len(max_relative_residual) // 2])
order_fa = np.argsort(fa_grid[example_i])
fa_min, fa_max = fa_grid[example_i].min(), fa_grid[example_i].max()
pad = 0.06 * (fa_max - fa_min)
dense_fa = np.linspace(fa_min - pad, fa_max + pad, 400)
dense_weights = quadratic_fa_spline_weights(
    np.full(len(dense_fa), q2_true[example_i]), dense_fa,
    np.repeat(ma_weights[example_i][None, :], len(dense_fa), axis=0),
)

fig, ax = plt.subplots(figsize=(7.0, 5.0), constrained_layout=True)
ax.plot(dense_fa, dense_weights, color=COLOR_FIT, linewidth=2.0, zorder=3,
        label=r'Quadratic fit in $F_A$')
ax.plot(fa_grid[example_i, order_fa], ma_weights[example_i, order_fa], linestyle='none',
        marker='o', markersize=6.5, color='black', zorder=4,
        label=r'GENIE $M_A$ spline samples ($M_A = 0.8$ to $1.4$ GeV)')
ax.set_xlabel(r'$F_A(Q^2_{\mathrm{true}})$')
ax.set_ylabel('Event weight')
ax.set_xlim(dense_fa[0], dense_fa[-1])
ax.legend(loc='best', fontsize=12)
ax.text(0.03, 0.04,
        rf'$Q^2_{{\mathrm{{true}}}} = {q2_true[example_i]:.3f}$ GeV$^2$'
        '\n'
        rf'Maximum relative residual: {100 * max_relative_residual[example_i]:.4f}%',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=12, color=COLOR_NOTE)
style_axis(ax)

save_figure(fig, 'quadratic_fa_representative_event')
plt.show()

## Summary of the numbers

In [ ]:
print(f'Fit sample: {n_selected:,} selected simulated events, {n_responsive:,} responsive '
      f'({100 * n_responsive / n_selected:.1f}%), tolerance on the weight range {RESPONSIVE_TOLERANCE:g}.')
print(f'Per-event max. relative residual: median {median_residual:.2e}, 99th pct {p99_residual:.2e}, '
      f'worst {max_relative_residual[worst]:.2e} (Q2_true = {q2_true[worst]:.3f} GeV^2).')
print(f'Residual vs Q2_true: median ~ Q2^{slope:.2f}; Spearman rho = {rank_corr:.2f}.')
print(f'Binned closure, largest |dS/S|: 2D fit bins {np.abs(closure_2d).max():.2e}, '
      f'log10 Q2_reco projection {np.abs(closure_q2).max():.2e}, p_n projection {np.abs(closure_pn).max():.2e}, '
      f'PROfit-style ratio to the central knot {np.abs(closure_2d_ratio).max():.2e}.')
print(f'Q2-shift refit: median residual {np.median(max_relative_residual):.2e} -> {np.median(shifted_residual):.2e}, '
      f'99th pct {np.percentile(max_relative_residual, 99):.2e} -> {np.percentile(shifted_residual, 99):.2e}; '
      f'delta = {lin[1]:.3f} + {lin[0]:.3f} Q2_true GeV^2.')